# Notebook 4: Real Data Backtest — AC2001 vs TWAP vs VWAP

Applies the AC2001 model to real equity data, following the methodology:

1. Fetch daily bars for liquid US equities via `yfinance`.
2. Estimate σ from historical data (close-to-close, GARCH, HAR-RV).
3. Simulate liquidating a position of X shares over T=1 day.
4. Compare realised IS (bps) for AC, TWAP, VWAP.
5. Report results in a summary table.

**Methodology note**: parameters η and γ are estimated using the Almgren et al. (2005) scaling relation rather than fitted to proprietary execution data.  This is stated explicitly; no overfitting to make AC look artificially better.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import yfinance as yf
    HAS_YFINANCE = True
except ImportError:
    HAS_YFINANCE = False
    print("yfinance not installed — using synthetic data.  Run: pip install yfinance")

from src.vol_estimator import close_to_close_vol, GARCH11, HARRV, log_returns
from src.market_impact import almgren_2005_params
from src.execution_sim import compare_strategies, slippage_summary

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Data Fetching

Using 2 years of daily bars for 5 liquid US equities.  If `yfinance` is not available, synthetic GBM data is used as a stand-in.

In [ ]:
TICKERS  = ['AAPL', 'MSFT', 'SPY', 'QQQ', 'TSLA']
N_SIMS   = 500
X_SHARES = 500_000     # position size (shares)
T_DAYS   = 1.0
N_INTERVALS = 390
LAM      = 1e-6
SEED     = 0

def fetch_prices(ticker: str, period: str = '2y') -> pd.Series:
    """Fetch adjusted close prices, or fall back to synthetic GBM."""
    if HAS_YFINANCE:
        try:
            data = yf.download(ticker, period=period, auto_adjust=True, progress=False)
            return data['Close'].dropna()
        except Exception as e:
            print(f"  yfinance error for {ticker}: {e}")

    # Synthetic fallback: GBM with reasonable parameters
    rng = np.random.default_rng(hash(ticker) % 2**32)
    n = 504  # ~2 years
    daily_ret = rng.normal(0.0003, 0.02, size=n)
    prices = 100.0 * np.exp(np.cumsum(daily_ret))
    return pd.Series(prices, name=ticker)


price_data = {}
for ticker in TICKERS:
    prices = fetch_prices(ticker)
    price_data[ticker] = prices
    print(f"{ticker:5s}: {len(prices):4d} daily bars  |  last price: ${prices.iloc[-1]:.2f}")

## 2. Volatility Estimation

Three estimators per ticker:
- **CC**: 21-day close-to-close std of log returns
- **GARCH**: current conditional vol from GARCH(1,1)
- **HAR-RV**: 1-step-ahead forecast from HAR-RV

In [ ]:
vol_results = []

for ticker, prices in price_data.items():
    p = prices.values.astype(float)
    r = log_returns(p)
    rv = r**2

    cc = close_to_close_vol(p, window=21)
    cc_scalar = float(cc[-1]) if hasattr(cc, '__len__') else float(cc)

    try:
        g = GARCH11().fit(p)
        garch_vol = g.daily_vol
        garch_unc = g.unconditional_vol
    except Exception:
        garch_vol = garch_unc = np.nan

    try:
        har = HARRV().fit(rv)
        har_vol = har.daily_vol
    except Exception:
        har_vol = np.nan

    vol_results.append({
        'Ticker':         ticker,
        'CC vol (21d)':   round(cc_scalar * 100, 3),
        'GARCH vol':      round(garch_vol * 100, 3),
        'GARCH uncond':   round(garch_unc * 100, 3),
        'HAR-RV vol':     round(har_vol * 100, 3),
        '_sigma':         cc_scalar,          # use CC vol for backtest
        '_price':         float(prices.iloc[-1]),
    })

vol_df = pd.DataFrame(vol_results)
display_df = vol_df[['Ticker', 'CC vol (21d)', 'GARCH vol', 'GARCH uncond', 'HAR-RV vol']]
print(display_df.to_string(index=False))

In [ ]:
# Volatility comparison bar chart
plot_df = display_df.set_index('Ticker')
plot_df.plot(kind='bar', figsize=(10, 4.5), colormap='tab10', rot=0)
plt.ylabel('Daily vol (%)')
plt.title('Volatility Estimates: CC vs GARCH vs HAR-RV')
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('../data/fig_vol_comparison.png', dpi=150)
plt.show()

## 3. Impact Parameter Estimation

Using the Almgren et al. (2005) scaling relation.  ADV is assumed at 0.5% of shares outstanding (a rough heuristic).  These parameters should be treated as order-of-magnitude estimates.

In [ ]:
# Approximate ADV (shares) for each ticker — rough order-of-magnitude
ADV_APPROX = {
    'AAPL':  60_000_000,
    'MSFT':  25_000_000,
    'SPY':  100_000_000,
    'QQQ':   50_000_000,
    'TSLA':  80_000_000,
}

impact_params = []
for row in vol_results:
    ticker = row['Ticker']
    sigma_dollar = row['_sigma'] * row['_price']
    adv = ADV_APPROX.get(ticker, 20_000_000)
    p = almgren_2005_params(sigma=sigma_dollar, adv=adv, price=row['_price'])
    impact_params.append({
        'Ticker': ticker,
        'η (×10⁻⁹)': round(p['eta'] * 1e9, 4),
        'γ (×10⁻⁹)': round(p['gamma'] * 1e9, 4),
        '_eta':       p['eta'],
        '_gamma':     p['gamma'],
    })

imp_df = pd.DataFrame(impact_params)
print(imp_df[['Ticker', 'η (×10⁻⁹)', 'γ (×10⁻⁹)']].to_string(index=False))

## 4. Backtest: IS Comparison per Ticker

In [ ]:
backtest_results = []

for vrow, irow in zip(vol_results, impact_params):
    ticker  = vrow['Ticker']
    sigma   = vrow['_sigma']
    price   = vrow['_price']
    eta     = irow['_eta']
    gamma   = irow['_gamma']

    print(f"  Running {ticker} ({N_SIMS} sims)...", end=' ')
    stats = compare_strategies(
        X=X_SHARES, T=T_DAYS, N=N_INTERVALS,
        sigma=sigma, gamma=gamma, eta=eta, lam=LAM,
        S0=price, n_sims=N_SIMS, seed=SEED,
    )
    print('done')

    for name, s in stats.items():
        backtest_results.append({
            'Ticker':       ticker,
            'Strategy':     name,
            'Mean IS (bps)': round(s.mean_is_bps, 2),
            'Std IS (bps)':  round(s.std_is_bps, 2),
            'Sharpe IS':     round(s.sharpe_is, 3),
            '_is_all':       s.is_bps_all,
        })

bt_df = pd.DataFrame(backtest_results)

In [ ]:
# Summary table
summary = bt_df[['Ticker', 'Strategy', 'Mean IS (bps)', 'Std IS (bps)', 'Sharpe IS']]
pivot = summary.pivot_table(
    index='Ticker',
    columns='Strategy',
    values=['Mean IS (bps)', 'Std IS (bps)'],
)
print(pivot.to_string())

## 5. Visualisation: IS Distribution by Strategy and Ticker

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS), figsize=(16, 5), sharey=False)

for ax, ticker in zip(axes, TICKERS):
    ticker_rows = [r for r in backtest_results if r['Ticker'] == ticker]
    data   = [r['_is_all'] for r in ticker_rows]
    labels = [r['Strategy'] for r in ticker_rows]

    ax.boxplot(data, labels=labels, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               medianprops=dict(color='black', lw=2))
    ax.set_title(ticker, fontsize=12)
    ax.set_ylabel('IS (bps)' if ax == axes[0] else '')
    ax.tick_params(axis='x', labelsize=9)

fig.suptitle(f'Implementation Shortfall Distribution — {N_SIMS} simulations per ticker',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../data/fig_backtest_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Caveats and Limitations

1. **Impact parameters**: η and γ are derived from the Almgren et al. (2005) cross-sectional regression.  They are order-of-magnitude estimates; actual impact varies by broker, time of day, and market regime.

2. **Volatility**: using close-to-close daily vol is a rough proxy.  Intraday vol is typically higher at open and close; the model uses a constant σ throughout the day.

3. **Price model**: the AC2001 GBM dynamics are used both to *design* the trajectory and to *simulate* execution.  In a real backtest, out-of-sample price paths (e.g. from tick data) should be used for the simulation stage.

4. **Position size**: $X = 500\,000$ shares is large relative to ADV for some tickers.  In practice this would require multi-day execution or a different impact regime.

5. **VWAP profile**: the U-shaped profile is stylised.  Real VWAP benchmarks use observed volume data.

These limitations are why the comparison is simulation-based rather than a live-trading P&L record.